# 03 Summarization

Qualitative analysis for §2 multi-granularity summarization. Run the extractive, abstractive, tiny, and evaluation commands in the README first so `results/summary_eval.csv` and prediction CSVs exist.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/multilexsum_clean.parquet')
PRED = Path('../results/abstractive_summaries.csv')
EVAL = Path('../results/summary_eval.csv')
LLM = Path('../results/llm_baseline_summaries.csv')

df = pd.read_parquet(DATA)
pred = pd.read_csv(PRED) if PRED.exists() else pd.DataFrame()
scores = pd.read_csv(EVAL) if EVAL.exists() else pd.DataFrame()
df.shape, pred.shape, scores.shape

## Length Reduction

Verify that extractive reduction reaches the W8 target of median reduction >= 90%.

In [ ]:
lengths_path = Path('../results/extractive_lengths.csv')
lengths = pd.read_csv(lengths_path) if lengths_path.exists() else pd.DataFrame()
if not lengths.empty:
    display(lengths[['original_tokens', 'reduced_tokens', 'reduction_pct']].describe())
    print('median reduction:', round(lengths['reduction_pct'].median() * 100, 1), '%')
else:
    print('Run: python -m src.summarize.extractive --output results/extractive_lengths.csv')

## Aggregate Scores

In [ ]:
if not scores.empty:
    metric_cols = [c for c in scores.columns if c not in {'case_id', 'granularity'}]
    display(scores.groupby('granularity')[metric_cols].mean().round(4))
else:
    print('Run: python -m src.evaluate --predictions results/abstractive_summaries.csv')

## 5 Good Cases

Good cases are the highest ROUGE-L examples among test-set long summaries; inspect whether the model preserves parties, claims, and procedural posture.

In [ ]:
def show_case(case_id, granularity='long'):
    row = df[df.case_id == case_id].iloc[0]
    pred_row = pred[pred.case_id == case_id].iloc[0]
    print('CASE:', case_id)
    print('\nREFERENCE\n', row[f'{granularity}_ref'][:1800])
    print('\nPREDICTION\n', pred_row[f'{granularity}_pred'][:1800])

if not scores.empty and not pred.empty:
    good_ids = (scores[scores.granularity == 'long']
                .sort_values('rougeL', ascending=False)
                .head(5)['case_id'].tolist())
    for cid in good_ids:
        show_case(cid, 'long')
        print('\n' + '='*100 + '\n')

## 5 Likely Hallucination Cases

Start with the lowest ROUGE-L cases, then manually mark hallucinations where the prediction introduces unsupported parties, outcomes, statutes, or remedies.

In [ ]:
if not scores.empty and not pred.empty:
    bad_ids = (scores[scores.granularity == 'long']
               .sort_values('rougeL', ascending=True)
               .head(5)['case_id'].tolist())
    for cid in bad_ids:
        show_case(cid, 'long')
        print('\nManual hallucination notes:')
        print('- Unsupported fact/outcome:')
        print('- Missing key claim/procedure:')
        print('\n' + '='*100 + '\n')